# KCA PDF Graph RAG

`0417_kca_split.pdf` contains graph/table-heavy pages, so this notebook uses Upstage Document Parse instead of plain `PyPDFLoader`. Upstage returns Markdown-like page content that keeps more figure/table context for retrieval.

In [9]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv


def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr


PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("LANGSMITH_TRACING", "false")

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "pdf" / "0417_kca_split.pdf"
INDEX_PATH = PROJECT_ROOT / "data" / "processed" / "kca_upstage_faiss"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PDF exists: {PDF_PATH.exists()} -> {PDF_PATH}")

PROJECT_ROOT: c:\Users\user\catcher\catcher-llm
PDF exists: True -> c:\Users\user\catcher\catcher-llm\data\raw\pdf\0417_kca_split.pdf


## 1. Parse the PDF with Upstage

Set `UPSTAGE_API_KEY` in `.env` first. Install the optional dependency group if needed:

```powershell
uv sync --group upstage
```

In [10]:
if not os.getenv("UPSTAGE_API_KEY"):
    raise RuntimeError(
        "UPSTAGE_API_KEY is missing. Add it to .env before parsing graph-heavy PDFs."
    )

from langchain_upstage import UpstageDocumentParseLoader


def load_with_upstage(pdf_path: Path):
    try:
        loader = UpstageDocumentParseLoader(
            str(pdf_path),
            split="page",
            output_format="markdown",
            coordinates=True,
        )
    except TypeError:
        # Older langchain-upstage versions accept fewer constructor options.
        loader = UpstageDocumentParseLoader(str(pdf_path), split="page")
    return loader.load()


docs = load_with_upstage(PDF_PATH)
print(f"Loaded pages: {len(docs)}")
print(docs[0].metadata)
print(docs[0].page_content[:800])

Loaded pages: 57
{'page': 1, 'coordinates': [[{'x': 0.1271, 'y': 0.0665}, {'x': 0.3576, 'y': 0.0665}, {'x': 0.3576, 'y': 0.0812}, {'x': 0.1271, 'y': 0.0812}], [{'x': 0.8307, 'y': 0.0659}, {'x': 0.866, 'y': 0.0659}, {'x': 0.866, 'y': 0.0826}, {'x': 0.8307, 'y': 0.0826}], [{'x': 0.1264, 'y': 0.1173}, {'x': 0.5644, 'y': 0.1173}, {'x': 0.5644, 'y': 0.1415}, {'x': 0.1264, 'y': 0.1415}], [{'x': 0.1304, 'y': 0.1605}, {'x': 0.8684, 'y': 0.1605}, {'x': 0.8684, 'y': 0.481}, {'x': 0.1304, 'y': 0.481}], [{'x': 0.1268, 'y': 0.5092}, {'x': 0.8716, 'y': 0.5092}, {'x': 0.8716, 'y': 0.5633}, {'x': 0.1268, 'y': 0.5633}], [{'x': 0.1449, 'y': 0.5814}, {'x': 0.8719, 'y': 0.5814}, {'x': 0.8719, 'y': 0.6923}, {'x': 0.1449, 'y': 0.6923}], [{'x': 0.1452, 'y': 0.7126}, {'x': 0.8716, 'y': 0.7126}, {'x': 0.8716, 'y': 0.7947}, {'x': 0.1452, 'y': 0.7947}], [{'x': 0.1266, 'y': 0.8742}, {'x': 0.6546, 'y': 0.8742}, {'x': 0.6546, 'y': 0.8918}, {'x': 0.1266, 'y': 0.8918}]]}
PART 1_제5장 2023 가계소비 현황과 인식 761 제2절 비대면 디지털시대 

## 2. Mark pages that likely contain visual content

This keeps graph/table-heavy chunks searchable and easy to inspect after retrieval.

In [11]:
VISUAL_KEYWORDS = (
    "\uadf8\ub9bc",
    "\ud45c",
    "chart",
    "figure",
    "table",
    "base:",
    "\ub2e8\uc704:",
    "%",
)

for doc in docs:
    content = doc.page_content.lower()
    doc.metadata["source"] = str(PDF_PATH)
    doc.metadata["parser"] = "upstage_document_parse"
    doc.metadata["has_visual_hint"] = any(keyword.lower() in content for keyword in VISUAL_KEYWORDS)

visual_docs = [doc for doc in docs if doc.metadata["has_visual_hint"]]
print(f"Visual-like pages: {len(visual_docs)} / {len(docs)}")
for doc in visual_docs[:5]:
    page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
    print(f"page={page} | {doc.page_content[:160].replace(chr(10), ' ')}")

Visual-like pages: 57 / 57
page=1 | PART 1_제5장 2023 가계소비 현황과 인식 761 제2절 비대면 디지털시대 가계소비 모습 | 지표 | 디지털 소비생활20) | 디지털 소비와 결제수단 이용 현황 | | --- | --- | --- | | 10-64 | 디지털 소비생활20) | 디지털 소비와 결제수단 이용 현황 |
page=2 | 762 2023 한국의 소비생활지표 □ (분야 현황) 디지털결제 수단을 통해 주로 소비한 분야는 '식품·외식'(92.7%), '의류'(36.4%), '문화·여가'(18.8%), '생활위생·미용'(18.0%) 등의 순으로 나타남21)(1 +2 + 3순위 기준) - ○ (소비자 특성별) 디
page=3 | PART 1_제5장 2023 가계소비 현황과 인식 763 【그림 5-2-1】 디지털결제 수단 이용률 (Base: 전체, 단위: %) ![image](/image/placeholder) - Chart Type: pie |  | 없다 | 있다 | 연간 평균 이용횟수 | | --- | ---
page=4 | 764 2023 한국의 소비생활지표 【그림 5-2-3】 전자상거래 경험별 디지털결제 수단으로 주로 소비한 품목 (1+2+3순위 기준) (Base: 디지털결제 수단 이용자, n=4,993명, 단위: %) | 전자상거래 경험 | 전자상거래 경험 | 전자상거래 경험 | 전자상거래 경험 | 전
page=5 | PART 1_제5장 2023 가계소비 현황과 인식 765 【그림 5-2-4】 연령별 디지털결제 수단으로 주로 소비한 품목(1 +2+3순위 기준) (Base: 디지털결제 수단 이용자, n=4,993명, 단위: %) ![image](/image/placeholder) 20대  92.4  4


## 3. Build the vector index

Chunking after page-level parsing helps retrieve only the graph/table section instead of an entire page.

In [12]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = int(os.getenv("RAG_CHUNK_SIZE", "1000"))
chunk_overlap = int(os.getenv("RAG_CHUNK_OVERLAP", "150"))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n## ", "\n### ", "\n\n", "\n", " ", ""],
)
chunks = splitter.split_documents(docs)

embeddings = OpenAIEmbeddings(model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"))
vectorstore = FAISS.from_documents(chunks, embeddings)
INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(INDEX_PATH))

print(f"Chunks: {len(chunks)}")
print(f"Vectors: {vectorstore.index.ntotal}")
print(f"Saved to: {INDEX_PATH}")

Chunks: 219
Vectors: 219
Saved to: c:\Users\user\catcher\catcher-llm\data\processed\kca_upstage_faiss


In [13]:
vectorstore = FAISS.load_local(
    str(INDEX_PATH),
    embeddings,
    allow_dangerous_deserialization=True,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": int(os.getenv("RAG_TOP_K", "4"))})

## 4. Test retrieval for graph questions

In [14]:
query = "\uc628\ub77c\uc778 \uac70\ub798 \ubd84\uc7c1\uc5d0\uc11c \uc8fc\ub85c \uc18c\ube44\ud558\ub294 \ubd84\uc57c\ub294 \ubb34\uc5c7\uc778\uac00? \uadf8\ub798\ud504 \uadfc\uac70\ub85c \uc54c\ub824\uc918"

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):
    page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
    has_visual = doc.metadata.get("has_visual_hint")
    print(f"[{i}] page={page} visual={has_visual}")
    print(doc.page_content[:700])
    print("=" * 80)

[1] page=57 visual=True
| 개인간 (C2C) 거래 플랫폼 쇼핑 | (256) | 3.5 | 28.1 | 39.1 | 4.3 | 10.9 | 3.9 | 13.7 | 4.7 | 8.2 | 2.0 | 5.5 | 12.1 | 23.8 |  | 3.5 | 9.0 | 28.5 40.2 |  | 4.3 | 11.3 | 2.0 | 5.1 | 5.1 | 12.1 | 1.6 | 3.5 |
| 금융 플랫폼 | (83) | 1.1 | 7.2 | 13.3 | 2.4 | 3.6 | 1.2 | 13.3 | 7.2 | 13.3 | 2.4 | 6.0 | 4.8 |  | 13.3 | 4.8 |  | 20.5 28.9 |  | 30.1 | 43.4 | 13.3 | 26.5 | 9.6 | 21.7 | 1.2 | 6.0 |
| 해외 직구 | (223) | 3.1 | 14.3 | 19.7 | 2.2 | 9.0 | 3.1 | 10.8 | 3.1 | 5.8 | 0.4 | 5.8 | 12.1 | 26.5 |  | 3.1 10.3 |  | 4.9 14.8 |  | 0.4 | 5.4 | 0.4 | 3.6 | 3.6 | 7.6 | 52.0 56.5 |  |
| 라이브 커머스 | (41) | 0.6 | 17.1 | 29.3 | 12.2 | 17.1 | 9.8 | 19.5 | - | 4.9 | - | 4.9 | 4.9 | 7.3 | 2.4 | 4.9 | 2.4 | 7.3 | 4.9 | 7.3 | 9.8 | 1
[2] page=56 visual=True
PART 1_제6장 2023 한국의 소비생활 변화와 전망 827 ○ (거래유형별) 비대면 거래유형 중 소비자문제경험률이 가장 높은 유형은
'해외직구'로 2021년과 동일하나 경험률은 38.1%p 감소('21년 60.0% →
'23년 21.9%) - - 2021년에 비해 전반적으로 거래유형별 소비자문제경험률은 감소, 하락폭은
- 모바일쇼핑(38.7%p ↓), 해외직구(38.1%p ↓), 인터넷쇼핑(34.6%p ↓ ) 등의 순
 - - 특히, '23

## 5. Optional answer generation

The answer is constrained to retrieved chunks and includes page metadata so graph evidence can be checked.

In [15]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"), temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer in Korean using only the provided context. "
            "When the context is from a chart/table, summarize the trend or ranking and cite page metadata.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_docs(retrieved_docs):
    formatted = []
    for doc in retrieved_docs:
        page = doc.metadata.get("page", doc.metadata.get("page_number", "?"))
        formatted.append(f"[page={page}]\n{doc.page_content}")
    return "\n\n".join(formatted)


context = format_docs(results)
answer = (prompt | llm).invoke({"question": query, "context": context})
print(answer.content)

온라인 거래 분쟁에서 주로 소비되는 분야는 개인간(C2C) 거래 플랫폼 쇼핑입니다. 표와 그래프에 따르면 개인간 거래 플랫폼 쇼핑이 거래 유형 중 소비자 문제 경험률이 높고, 특히 상품·서비스 품질 불량(55.6%), 오배송 및 배송지연(40.2%), 거짓·과장·기만 표시광고(28.8%) 등의 문제가 많이 발생하는 것으로 나타났습니다(출처: 57페이지, 56페이지).
